# 🧬 Genomics Multiagent System

In this notebook we create a Supervisor Agent that orchestrates **5** specialized genomics agents using the Strands framework:
1. **Research Assistant Agent** - Queries and interprets somatic variant data (the agent YOU built in Modules 1 & 2)
2. **Data Discovery Agent** - Finds genomics data files (FASTQ pairs) for samples
3. **QC Agent** - Analyzes FASTQC quality control reports
4. **Workflow Orchestrator Agent** - Manages HealthOmics variant calling workflows
5. **Interpretation and Reporting Agent** - Provides cancer variant analysis with CIViC database integration


#### Install required dependencies

In [1]:
from IPython.display import HTML

HTML("""
<style>
/* Make all JupyterLab output areas scrollable */
.jp-OutputArea-output {
    max-height: 300px;
    overflow-y: auto;
}
</style>
""")

In [2]:
# Download and extract Module 3 dependencies
import urllib.request
import zipfile
import os

url = "https://ws-assets-prod-iad-r-iad-ed304a55c2ca1aee.s3.us-east-1.amazonaws.com/9985a25c-454f-455e-a341-bc5db8d02280/module3-dependencies.zip"
zip_path = "/tmp/module3-dependencies.zip"

urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(os.getcwd())

print("✅ Module 3 dependencies downloaded and extracted!")
print("Files:", [f for f in os.listdir('.') if f.endswith('.py') or f == 'requirements.txt'])

✅ Module 3 dependencies downloaded and extracted!
Files: ['workflow_orchestrator_agent.py', 'data_discovery_agent.py', 'mcp_clients.py', 'requirements.txt', 'research_assistant_agent.py', 'qc_agent.py', 'interpretation_and_reporting_agent.py']


In [3]:
!pip install pandas --upgrade --quiet
!pip install -r requirements.txt --quiet
!pip install uv --quiet

#### Import required libraries

We will also import the functions that create each of the five specialized agents:
- [`research_assistant_agent.py`](research_assistant_agent.py), your Research Assistant from Modules 1 & 2
- [`data_discovery_agent.py`](data_discovery_agent.py)
- [`qc_agent.py`](qc_agent.py)
- [`workflow_orchestrator_agent.py`](workflow_orchestrator_agent.py)
- [`interpretation_and_reporting_agent.py`](interpretation_and_reporting_agent.py)

In [4]:
import boto3
import logging
from strands import Agent, tool
from strands.models import BedrockModel

# Import agent creation functions
from data_discovery_agent import create_data_discovery_agent
from qc_agent import create_qc_agent
from workflow_orchestrator_agent import create_healthomics_agent
from interpretation_and_reporting_agent import create_cancer_analysis_agent
from research_assistant_agent import create_research_assistant_agent

# Import MCP client helper
from mcp_clients import setup_mcp_clients

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Agent Creation
In this section we create the supervisor agent and wrap the four specialized agents as tools

### Agents as Tools with Strands Agents

"Agents as Tools" is an architectural pattern in AI systems where specialized AI agents are wrapped as callable functions (tools) that can be used by other agents. This creates a hierarchical structure where:

1. A primary "supervisor" agent handles user interaction and determines which specialized agent to call
2. Specialized "sub-agents" perform domain-specific tasks when called by the supervisor

This approach mimics human team dynamics, where a manager coordinates specialists, each bringing unique expertise to solve complex problems.

### Initialize MCP Tools
First, we'll initialize the HealthOmics MCP tools that will be shared across agents. This is more efficient than each agent starting it's own instance of the MCP and is safe in this case because the MCP is stateless.

In [5]:
# Initialize MCP clients and get tools
print("Initializing MCP clients...")
healthomics_client, aws_api_client = setup_mcp_clients()

# Enter the context managers to activate the clients
healthomics_client.__enter__()
aws_api_client.__enter__()

# Get tools from the clients
healthomics_tools = healthomics_client.list_tools_sync()
aws_tools = aws_api_client.list_tools_sync()

print(f"✓ Loaded {len(healthomics_tools)} HealthOmics tools")
print(f"✓ Loaded {len(aws_tools)} AWS API tools")

Initializing MCP clients...
The current AWS region is: us-east-1
🔌 MCP clients created successfully
✓ Loaded 77 HealthOmics tools
✓ Loaded 2 AWS API tools


### Create Specialized Agents
Now we'll instantiate each of the five specialized agents, connecting them to their MCP servers as needed. The agents are defined in:
- [`interpretation_and_reporting_agent.py`](interpretation_and_reporting_agent.py),
- [`data_discovery_agent.py`](data_discovery_agent.py),
- [`qc_agent.py`](qc_agent.py),
- [`workflow_orchestrator_agent.py`](workflow_orchestrator_agent.py),
- [`research_assistant_agent.py`](research_assistant_agent.py) 

Each provides it's own 'create' method to instantiate them. All five Agents are built with the Strands SDK and follow patterns we have already shown. Feel free to take a look at their definitions.

In [6]:
# Create the 4 specialized agents with appropriate tools
print("Creating specialized agents...")

data_discovery_agent = create_data_discovery_agent(healthomics_tools + aws_tools)
print("✓ Data Discovery Agent created")

qc_agent = create_qc_agent(aws_tools)
print("✓ QC Agent created")

workflow_orchestrator_agent = create_healthomics_agent(healthomics_tools + aws_tools)
print("✓ Workflow Orchestrator Agent created")

interpretation_agent = create_cancer_analysis_agent(aws_tools)
print("✓ Interpretation and Reporting Agent created")

research_assistant = create_research_assistant_agent(aws_tools)
print("✓ Research Assistant Agent created")

print("\nAll specialized agents ready!")

INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


Creating specialized agents...
✓ Data Discovery Agent created


INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


✓ QC Agent created
✓ Workflow Orchestrator Agent created
✓ Interpretation and Reporting Agent created
✓ Research Assistant Agent created

All specialized agents ready!


### Wrap Agents as Tools
Now we'll wrap each specialized agent as a tool that the supervisor can call. The supervisor autonomously decides which tool to use based on the the current conversation content, each tools description (docs), and the supervisor's system prompt.

In [7]:
def _safe_agent_text(response, agent_label: str) -> str:
    """Return the agent's text, never an empty string.

    Bedrock's Converse API rejects messages that contain a blank text content
    block. If a wrapped agent returns no text (for example it only made tool
    calls), we substitute a short non-empty message so the supervisor's
    conversation history stays valid.
    """
    text = str(response).strip()
    if not text:
        return f"The {agent_label} completed but returned no text output."
    return text

# 0: Research Assistant Agent Tool (the agent you built in Modules 1 & 2)
@tool
def research_assistant_tool(query: str) -> str:
    """
    Query and interpret somatic variant data from the Mutect2 pipeline results.
    Use this agent when users ask about variants, genes, mutations, or want a summary of variant data.

    Args:
        query: Request about variant data (e.g., "What variants are in TP53?", "Show high-impact variants")

    Returns:
        Variant query results and interpretation
    """
    try:
        response = research_assistant(query)
        return _safe_agent_text(response, "Research Assistant Agent")
    except Exception as e:
        return f"Error in research assistant: {str(e)}"

# 1: Data Discovery Agent Tool
@tool
def data_discovery_tool(query: str) -> str:
    """
    Find genomics data files (FASTQ pairs) for a given sample ID using AWS HealthOmics.
    Use this agent when you need to locate sequencing data files.

    Args:
        query: Request to find genomics data files (e.g., "Find FASTQ or BAM files for sample ABC123").

    Returns:
        Information about the located data files including S3 URIs
    """
    try:
        response = data_discovery_agent(query)
        return _safe_agent_text(response, "Data Discovery Agent")
    except Exception as e:
        return f"Error in data discovery: {str(e)}"

# 2: QC Agent Tool
@tool
def quality_control_tool(query: str) -> str:
    """
    Analyze FASTQC quality control reports and provide recommendations for genomics data preprocessing.
    Use this agent when you need to assess data quality or get QC recommendations.

    Args:
        query: Request to analyze quality control data (e.g., "Analyze QC report at s3://bucket/qc.zip"). Ensure the query contains an S3 URI.
    Returns:
        Quality control analysis results and recommendations
    """
    try:
        response = qc_agent(query)
        return _safe_agent_text(response, "Quality Control Agent")
    except Exception as e:
        return f"Error in quality control: {str(e)}"

# 3: Workflow Orchestrator Agent Tool
@tool
def workflow_orchestrator_tool(query: str) -> str:
    """
    Orchestrate AWS HealthOmics workflows. Identify which workflows to run, start workflows, monitor their status,
    and retrieve results. Use this agent when you need to run genomics analysis pipelines.

    Args:
        query: Request to find, create or manage workflows (e.g., "Find workflow to call variants for sample xyz").

    Returns:
        Workflow execution status and results
    """
    try:
        response = workflow_orchestrator_agent(query)
        return _safe_agent_text(response, "Workflow Orchestrator Agent")
    except Exception as e:
        return f"Error in workflow orchestration: {str(e)}"

# 4: Interpretation and Reporting Agent Tool
@tool
def interpretation_reporting_tool(query: str) -> str:
    """
    Provide comprehensive cancer variant analysis with CIViC database integration.
    Offers evidence-based clinical interpretations and therapeutic recommendations.
    Use this agent when you need to interpret variant calling results.

    Args:
        query: Request for variant interpretation (e.g., "Analyze variants in MAF file at s3://bucket/variants.maf")

    Returns:
        Clinical interpretation and therapeutic recommendations
    """
    try:
        response = interpretation_agent(query)
        return _safe_agent_text(response, "Interpretation and Reporting Agent")
    except Exception as e:
        return f"Error in interpretation: {str(e)}"

print("Agent tools created successfully!")

Agent tools created successfully!


### Create the Supervisor Agent
Finally, we'll create the supervisor agent that coordinates all specialized agents

In [8]:
# Define supervisor agent configuration
supervisor_instruction = """You are a Genomics Pipeline Supervisor Agent that coordinates specialized agents to help users with genomics analysis workflows.

You have access to 5 specialized agents:
1. **Data Discovery Agent** - Finds genomics data files (FASTQ pairs) for samples
2. **Quality Control Agent** - Analyzes FASTQC quality control reports and provides recommendations
3. **Workflow Orchestrator Agent** - Finds or creates AWS Healthomics workflows, appropriate IAM roles, and reference data to run the appropriate workflows
4. **Interpretation and Reporting Agent** - Provides cancer variant analysis with CIViC database integration
5. **Research Assistant Agent** - Queries and interprets somatic variant data from the Mutect2 pipeline results

Your role is to:
- Understand user queries and determine which agent(s) to engage
- Route questions to the appropriate specialized agent
- Coordinate multi-step workflows that require multiple agents
- Synthesize responses from multiple agents when needed
- Provide clear, consolidated answers to the user

Common workflows:
- **Research Assistant**: Route variant queries directly (e.g., "What variants are in TP53?", "Show high-impact variants")
- **Data Discovery → QC → Workflow → Interpretation**: Find data, check quality, run workflow and call variants if quality is good, then interpret MAF file and generate report
- **Research Assistant → Interpretation**: Query variants, then get clinical interpretation with CIViC annotations
- **Workflow → Interpretation**: Run variant calling, then interpret results
- **Single Agent**: Route simple queries to the appropriate specialist

When responding:
1. Briefly acknowledge the user's request
2. Explain which agent(s) you're engaging and why
3. Present the results from each agent clearly
4. Provide a concise summary with actionable next steps

Be helpful, clear, and efficient in coordinating the genomics analysis pipeline.
"""

# Define the model for supervisor
supervisor_model = BedrockModel(
    model_id="global.anthropic.claude-sonnet-5",
    # 💡 Try Opus 5.5 for deeper reasoning (slower, higher cost): "global.anthropic.claude-opus-5-5"
    max_tokens=4096
)

# Create the supervisor agent
supervisor = Agent(
    name="Genomics Pipeline Supervisor",
    model=supervisor_model,
    system_prompt=supervisor_instruction,
    tools=[
    research_assistant_tool,
    data_discovery_tool,
    quality_control_tool,
    workflow_orchestrator_tool,
    interpretation_reporting_tool
    ]
)

print("✓ Supervisor Agent created successfully!")
print("\nThe supervisor is ready to coordinate genomics analysis workflows.")

INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


✓ Supervisor Agent created successfully!

The supervisor is ready to coordinate genomics analysis workflows.


In [9]:
# Helper: safely ask the supervisor.
# Skips empty questions and cleans any blank turns from the conversation history,
# so an accidental empty run never breaks later queries.
def ask_supervisor(user_query: str):
    # Self-heal: drop any turns whose only content is blank text.
    supervisor.messages[:] = [
        m for m in supervisor.messages
        if any(
            (block.get("text") or "").strip() or "toolUse" in block or "toolResult" in block
            for block in m.get("content", [])
        )
    ]
    if not user_query.strip():
        print("✏️  Type a question into user_query above, then run this cell again.")
        return
    print(f"User: {user_query}")
    print("\n" + "=" * 80)
    response = supervisor(user_query)
    print(f"\nSupervisor: {response}")

print("✓ ask_supervisor() ready. Use it to query the supervisor safely.")

✓ ask_supervisor() ready. Use it to query the supervisor safely.


## Test the Supervisor Agent
Now let's test the supervisor with different types of queries. In the agent responses you will see it select various tools to help it formulate it's response.

### Sample Queries to Execute & Test

Here are some example questions you can ask the supervisor agent. Pick one to assign to `test_query` below, then run this cell to send it to the supervisor and see which specialized agent it routes to.

In [10]:
session = boto3.Session()
region = session.region_name
print(f"The current AWS region is: {region}")

# Sample question bank

# Data Discovery Questions
data_discovery_q1 = "Find the FASTQ files for patient P001"
data_discovery_q2 = "Which reference genomes are available to me?"

# Quality Control Questions
qc_q1 = "Is the data quality good enough to proceed with variant calling?"

# Workflow Questions
workflow_q1 = "Start a variant calling workflow for the FASTQ files I just found"
workflow_q2 = "Check the status of workflow run 1234567"
workflow_q3 = "List all my recent workflow runs"

# Interpretation Questions
interpretation_q1 = f"Analyze the variants in the MAF file at s3://aws-genomics-static-{region}/omics-data/tumor-normal/maf/test_civic.mafs"
interpretation_q2 = "Generate a clinical report for the variant analysis"

# Research Assistant Questions (the agent you built!)
research_q1 = "What variants are in TP53?"
research_q2 = "Show me the high-impact variants"
research_q3 = "Give me a summary of all variants in the dataset"

# Multi-step Questions
multi_step_q1 = "Find data for sample XYZ, check its quality, and if good, start variant calling"
multi_step_q2 = "Run variant calling on sample ABC and then interpret the results"

# Choose a question to test, then run this cell to send it to the supervisor.
test_query = research_q1  # Start with your Research Assistant! Change this to any question above.

print(f"Selected question: {test_query}")
print("=" * 80)

# Now Execute and Test the Query
ask_supervisor(test_query)

INFO:strands.telemetry.metrics:Creating Strands MetricsClient


The current AWS region is: us-east-1
Selected question: What variants are in TP53?
User: What variants are in TP53?


Tool #1: research_assistant_tool


INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole



Tool #1: query_variants_by_gene
## TP53 Variant Found

I found **1 variant** in the TP53 gene in this dataset:

| Feature | Detail |
|---|---|
| **Gene** | TP53 |
| **Variant Type** | Nonsense Mutation |
| **Protein Change** | p.R213* |
| **Location** | Chromosome 17, position 7,674,894 |
| **Read Depth** | 38 total reads |
| **Variant Allele Count** | 20 reads supporting the mutation |

### What this means in plain language:

- **Nonsense mutation**: This type of mutation creates a premature "stop" signal in the gene's instructions. Instead of producing a full, functional TP53 protein, the cell stops building the protein partway through.
- **p.R213***: At position 213 in the protein, an arginine (R) amino acid is replaced by a stop codon (indicated by the asterisk *). This truncates the protein, likely destroying its function.
- **Clinical significance**: TP53 is one of the most important tumor suppressor genes in the human genome (often called the "guardian of the genome"). It norma

### 🧪 DIY: Try Your Own Queries

Now it's your turn. Here are some ideas to explore with the multi-agent system:

**Idea 1: Cross-Agent Research Query**
Ask a question that requires multiple agents to collaborate:
- "Find the available data for patient P001, check the quality, show me the variants, and highlight anything clinically significant"
- "Run variant calling on the tumour-normal samples and then interpret the results"

**Idea 2: Compare Agent Responses**
Ask related questions that hit different agents and compare the depth of response:
- "What variants are in TP53?" (routes to Research Assistant)
- "What is the clinical significance of TP53 nonsense mutations in cancer?" (routes to Interpretation agent)
- "What variants are in TP53 and what do they mean clinically?" (should engage both)

Use the interactive cell below to try these. Change `user_query` and run it to see how the supervisor routes to different agents.

Add additional cells below the Interactive Query cell to try your own queries.

In [11]:
# Interactive query - modify and run multiple times
user_query = "Find the FASTQ files for patient P001"  # Change this to ask different questions

ask_supervisor(user_query)

User: Find the FASTQ files for patient P001


Tool #2: data_discovery_tool

Tool #1: SearchGenomicsFiles
I found **two complete paired-end FASTQ sets for patient P001**, representing a tumor-normal pair. Both are fully accessible (STANDARD/hot storage, no retrieval delay needed) and located in the same S3 bucket. Relevance score: 0.92 (excellent match) for both.

## 1. P001 — Tumor Sample (SRR2089363)
| | R1 | R2 |
|---|---|---|
| **Path** | `s3://aws-genomics-static-us-east-1/omics-data/tumor-normal/fastqs/SRR2089363_R1.fastq` | `s3://aws-genomics-static-us-east-1/omics-data/tumor-normal/fastqs/SRR2089363_R2.fastq` |
| **Size** | 1.9 GB | 1.9 GB |
| **Storage** | STANDARD (hot) | STANDARD (hot) |
| **Last Modified** | 2025-10-21 | 2025-10-21 |

- Tags: `sample=P001`, `sample-type=tumor`
- Paired-end set complete (R1/R2 confirmed via matching file sizes/naming)
- Combined size: ~3.9 GB

## 2. P001 — Normal Sample (SRR2089364)
| | R1 | R2 |
|---|---|---|
| **Path** | `s3://aws-genomics-

**Idea 3: Build Your Own Multi-Agent System**

Now take what you have learned and sketch a multi-agent system for your own work. In a new cell (or cells) below, use the same patterns from this notebook to:

1. Define or import the tools your specialists would need (the `@tool` functions, like the ones in the agent files you imported earlier)
2. Create each specialized agent with its own model, tools, and system prompt
3. Wrap each agent as a tool (the "agents as tools" pattern)
4. Define a supervisor agent that routes to them

Think about your day-to-day research: what are the specialists you would want on your team, and what would each one do? Use the **+** button in the toolbar to add new cells and start building.